In [ ]:
#@title 1. Tải repo, cài thư viện & nạp model F5-TTS Tiếng Việt
REPO_URL = "https://github.com/luuluyenmotdongfor/f5-tts-vi-share.git" #@param {type:"string"}

import os
import unicodedata

WORK_DIR = "/content" if os.path.exists("/content") else os.getcwd()
os.chdir(WORK_DIR)

if not os.path.exists("voice_vi_mau.wav"):
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    repo_dir = os.path.join(WORK_DIR, repo_name)
    if not os.path.exists(repo_dir):
        !git clone {REPO_URL}
    else:
        print(f"Thư mục {repo_dir} đã tồn tại, đang cập nhật...")
        !git -C {repo_dir} pull
    os.chdir(repo_dir)

print("Thư mục làm việc:", os.getcwd())
!nvidia-smi || echo "CẢNH BÁO: Chưa bật GPU! Vào Runtime -> Change runtime type -> T4 GPU."
!pip install -q f5-tts huggingface_hub soundfile ipywidgets

from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

CKPT_REPO = "hynt/F5-TTS-Vietnamese-ViVoice"
REF_WAV = os.path.abspath("voice_vi_mau.wav")
REF_TEXT = "lấy ngẫu nhiên từng quả ra cho đến khi hết, tính xác suất để lấy được tất cả 4 bóng đỏ, trước khi lấy được 3 bóng trắng. học sinh bình thường nhìn thấy bài này sẽ đâm đầu vào vẽ sơ đồ cây."

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower().strip()
    text = " ".join(text.split())
    if text and text[-1] not in ".!?":
        text += "."
    return text

def split_segments(raw: str):
    raw = raw.strip()
    if "|" in raw:
        return [" ".join(p.split()) for p in raw.split("|") if p.strip()]
    return [" ".join(p.split()) for p in raw.splitlines() if p.strip()]

print("Đang tải / kiểm tra checkpoint tiếng Việt từ Hugging Face...")
ckpt_file = hf_hub_download(repo_id=CKPT_REPO, filename="model_last.pt")
vocab_file = hf_hub_download(repo_id=CKPT_REPO, filename="config.json")

print("Đang nạp model F5TTS_Base lên GPU...")
tts = F5TTS(model="F5TTS_Base", ckpt_file=ckpt_file, vocab_file=vocab_file)
print(">>> SẴN SÀNG! Hãy bấm Play ở Cell 2 để mở bảng nhập thoại & nghe thử ngay.")

In [ ]:
#@title 2. Giao diện nhập thoại, nghe thử & tải audio trực tiếp trên Colab
import base64
import os
import zipfile
import numpy as np
import soundfile as sf
import ipywidgets as widgets
from IPython.display import Audio, HTML, clear_output, display

if "tts" not in globals():
    raise RuntimeError("Bạn cần chạy Cell 1 trước để tải model lên GPU!")

txt_box = widgets.Textarea(
    value="Xin chào, đây là giọng đọc trí tuệ nhân tạo tiếng Việt.\nMùa hè năm một nghìn không trăm chín mươi chín, kỵ sĩ En Xít lao ra khỏi cổng thành Va-len-xi-a.",
    placeholder="Nhập lời thoại tiếng Việt tại đây (ngăn cách nhiều đoạn bằng dấu | hoặc xuống dòng)...",
    layout=widgets.Layout(width="100%", height="140px"),
)
sld_speed = widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.05, description="Tốc độ:")
sld_pause = widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="Nghỉ câu (s):")
int_seed = widgets.IntText(value=42, description="Seed:", layout=widgets.Layout(width="180px"))
btn_gen = widgets.Button(description="🎙️ Tạo giọng đọc & Nghe thử", button_style="primary", layout=widgets.Layout(width="260px", height="40px"))
out_area = widgets.Output()

def _make_download_html(file_path, label):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    if size_mb > 15:
        return ""
    mime = "application/zip" if file_path.endswith(".zip") else "audio/wav"
    b64 = base64.b64encode(open(file_path, "rb").read()).decode("ascii")
    fname = os.path.basename(file_path)
    return f'<a download="{fname}" href="data:{mime};base64,{b64}" style="display:inline-block;padding:8px 16px;margin:4px 8px 4px 0;background:#16a34a;color:#fff;font-weight:bold;border-radius:6px;text-decoration:none;">{label} ({size_mb:.2f} MB)</a>'

def on_generate_click(_):
    global part_files, segments
    with out_area:
        clear_output(wait=True)
        segments = split_segments(txt_box.value)
        if not segments:
            print("⚠️ Vui lòng nhập ít nhất một câu tiếng Việt.")
            return
        print(f"Đang xử lý {len(segments)} đoạn thoại (tốc độ={sld_speed.value}, seed={int_seed.value})...")
        os.makedirs("outputs_vi", exist_ok=True)
        ref_norm = normalize_vi(REF_TEXT)
        part_files = []
        chunks = []
        out_sr = 24000

        for idx, seg in enumerate(segments, 1):
            seg_norm = normalize_vi(seg)
            p_path = os.path.join("outputs_vi", f"part_{idx:03d}.wav")
            print(f"  [{idx}/{len(segments)}] {seg_norm}")
            tts.infer(
                ref_file=REF_WAV,
                ref_text=ref_norm,
                gen_text=seg_norm,
                file_wave=p_path,
                speed=float(sld_speed.value),
                seed=int(int_seed.value),
                remove_silence=False,
            )
            part_files.append(p_path)
            data, sr = sf.read(p_path)
            if data.ndim > 1:
                data = data.mean(axis=1)
            out_sr = sr
            chunks.append(data.astype(np.float32))
            if idx < len(segments) and sld_pause.value > 0:
                chunks.append(np.zeros(int(sld_pause.value * sr), dtype=np.float32))

        merged = np.concatenate(chunks)
        sf.write("output.wav", merged, out_sr)
        dur = len(merged) / float(out_sr)

        zip_path = None
        if len(part_files) > 1:
            zip_path = "f5_tts_vi_outputs.zip"
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
                zf.write("output.wav")
                for pf in part_files:
                    zf.write(pf)

        clear_output(wait=True)
        print(f"✅ Hoàn tất {len(part_files)} đoạn — Tổng thời lượng: {dur:.2f} giây")
        print("🎧 Nghe thử bản hoàn chỉnh (output.wav):")
        display(Audio("output.wav", rate=out_sr))

        html_links = _make_download_html("output.wav", "⬇️ Tải file ghép (.wav)")
        if zip_path:
            html_links += _make_download_html(zip_path, "📦 Tải trọn bộ từng đoạn (.zip)")
        if html_links:
            display(HTML(html_links))

        btn_colab_dl = widgets.Button(description="💾 Hoặc bấm tải qua Colab", button_style="success", layout=widgets.Layout(width="220px"))
        def _dl_colab(_btn):
            from google.colab import files
            files.download(zip_path if zip_path else "output.wav")
        btn_colab_dl.on_click(_dl_colab)
        display(btn_colab_dl)

btn_gen.on_click(on_generate_click)
display(widgets.VBox([
    widgets.HTML("<b>Nhập văn bản tiếng Việt cần đọc</b> (Mẹo: viết số bằng chữ, phiên âm mềm tên tiếng Anh):"),
    txt_box,
    widgets.HBox([sld_speed, sld_pause, int_seed]),
    btn_gen,
    out_area,
]))

In [ ]:
#@title 3. (Tuỳ chọn) Chế độ Form nhanh nếu muốn dán 1 dòng và bấm Play
KICH_BAN = "Xin chào, đây là giọng đọc trí tuệ nhân tạo tiếng Việt. | Mùa hè năm một nghìn không trăm chín mươi chín, kỵ sĩ En Xít lao ra khỏi cổng thành Va-len-xi-a." #@param {type:"string"}
TOC_DO = 1.0 #@param {type:"slider", min:0.7, max:1.3, step:0.05}
SEED = 42 #@param {type:"integer"}

import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

segments = split_segments(KICH_BAN)
if not segments:
    raise ValueError("Kịch bản đang trống.")

os.makedirs("outputs_vi", exist_ok=True)
ref_norm = normalize_vi(REF_TEXT)
part_files = []
chunks = []
out_sr = 24000

for idx, seg in enumerate(segments, 1):
    seg_norm = normalize_vi(seg)
    p_path = os.path.join("outputs_vi", f"part_{idx:03d}.wav")
    print(f"[{idx}/{len(segments)}] Đang sinh: {seg_norm}")
    tts.infer(
        ref_file=REF_WAV,
        ref_text=ref_norm,
        gen_text=seg_norm,
        file_wave=p_path,
        speed=float(TOC_DO),
        seed=int(SEED),
        remove_silence=False,
    )
    part_files.append(p_path)
    data, sr = sf.read(p_path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    out_sr = sr
    chunks.append(data.astype(np.float32))
    if idx < len(segments):
        chunks.append(np.zeros(int(0.25 * sr), dtype=np.float32))

merged = np.concatenate(chunks)
sf.write("output.wav", merged, out_sr)
print(f"\n>>> Hoàn tất ({len(merged)/out_sr:.2f} giây)! Nghe thử bên dưới:")
display(Audio("output.wav", rate=out_sr))

In [ ]:
#@title 4. Tải file âm thanh về máy (.wav hoặc .zip nếu nhiều đoạn)
import os
import zipfile
from google.colab import files

if not os.path.exists("output.wav"):
    raise FileNotFoundError("Chưa có file output.wav. Hãy tạo giọng ở Cell 2 hoặc Cell 3 trước.")

if "part_files" not in globals() or len(part_files) <= 1:
    print("Đang tải về output.wav...")
    files.download("output.wav")
else:
    zip_name = "f5_tts_vi_outputs.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("output.wav")
        for pf in part_files:
            if os.path.exists(pf):
                zf.write(pf)
    print(f"Đã nén {len(part_files)} đoạn và bản ghép output.wav vào {zip_name}. Đang tải về...")
    files.download(zip_name)